# U3 – Análisis supervisado: predicción del VEI de erupciones volcánicas
**Equipo 6 · Grupo 9IDGSB · Universidad Tecnológica de Coahuila**

Notebook que acompaña el documento *U3 Saber – Análisis supervisado*. Usa el archivo `erupciones_plano.csv` preparado en la unidad anterior (fuente: Smithsonian Global Volcanism Program).

**Dos problemas supervisados sobre la misma tabla:**
1. **Regresión:** estimar el VEI (0–7) de una erupción.
2. **Clasificación binaria:** VEI ≥ 3 (positivo) frente a VEI < 3.

**Variables de entrada (X):** `latitud`, `longitud`, `elevacion_m`, `anio_inicio`.
**Excluidas por fuga de la respuesta:** `vei_modificador`, `categoria_explosividad`, `vei_incierto`.

**Flujo:** filtrar → dividir 80/20 → (en cada modelo: imputar mediana → estandarizar → algoritmo, todo ajustado solo con entrenamiento) → validación cruzada de 5 pliegues → optimización de hiperparámetros → ajuste de umbral (clasificación) → **evaluación única en prueba**.

In [1]:
import warnings, os, json
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                     cross_validate, cross_val_predict, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              RandomForestClassifier, GradientBoostingClassifier)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
os.makedirs('figuras', exist_ok=True)
os.makedirs('resultados', exist_ok=True)
plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
VERDE, NARANJA, GRIS, AZUL = '#0e8074', '#d9922b', '#8a8f98', '#2f5d8a'

## 1. Datos y planteamiento

In [2]:
df = pd.read_csv('erupciones_plano.csv')
print('Registros totales:', len(df))

# Solo erupciones con VEI conocido y no incierto
d = df[df['vei'].notna() & (df['vei_incierto'] == 0)].reset_index(drop=True)
print('Registros con VEI conocido y no incierto:', len(d))

FEATURES = ['latitud', 'longitud', 'elevacion_m', 'anio_inicio']
X = d[FEATURES]
y_reg = d['vei']                      # regresión
y_clf = (d['vei'] >= 3).astype(int)   # clasificación: 1 = VEI >= 3

print('\nFaltantes en predictores:'); print(X.isna().sum().to_string())
print('\nDistribución del VEI:'); print(d['vei'].value_counts().sort_index().to_string())
print(f'\nProporción de clase positiva (VEI>=3): {y_clf.mean():.1%}')

Registros totales: 9918
Registros con VEI conocido y no incierto: 6840

Faltantes en predictores:
latitud         4
longitud        4
elevacion_m    12
anio_inicio     0

Distribución del VEI:
vei
0.0     789
1.0    1139
2.0    3285
3.0     975
4.0     450
5.0     154
6.0      43
7.0       5

Proporción de clase positiva (VEI>=3): 23.8%


In [3]:
# División 80/20, misma para ambas tareas, estratificada por la clase VEI>=3 (semilla fija)
idx_tr, idx_te = train_test_split(d.index, test_size=0.2, random_state=SEED, stratify=y_clf)
Xtr, Xte = X.loc[idx_tr], X.loc[idx_te]
ytr_r, yte_r = y_reg.loc[idx_tr], y_reg.loc[idx_te]
ytr_c, yte_c = y_clf.loc[idx_tr], y_clf.loc[idx_te]
print(f'Entrenamiento: {len(Xtr)} | Prueba: {len(Xte)} | Positivos en prueba: {yte_c.sum()}')

def pipe(modelo, escalar=True):
    pasos = [('imp', SimpleImputer(strategy='median'))]
    if escalar: pasos.append(('esc', StandardScaler()))
    pasos.append(('m', modelo))
    return Pipeline(pasos)

Entrenamiento: 5472 | Prueba: 1368 | Positivos en prueba: 325


## 2. Regresión: estimar el VEI
### 2.1 Modelos candidatos y validación cruzada (solo entrenamiento)

In [4]:
cv_r = KFold(n_splits=5, shuffle=True, random_state=SEED)
cand_r = {
    'Línea base (mediana)': pipe(DummyRegressor(strategy='median')),
    'Ridge':                pipe(Ridge(alpha=1.0)),
    'Random Forest':        pipe(RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1), escalar=False),
    'Gradient Boosting':    pipe(GradientBoostingRegressor(random_state=SEED), escalar=False),
}
def cv_reg(m):
    r = cross_validate(m, Xtr, ytr_r, cv=cv_r, n_jobs=-1,
                       scoring={'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'})
    return {'MAE': -r['test_mae'].mean(), 'RMSE': -r['test_rmse'].mean(), 'R2': r['test_r2'].mean(),
            'RMSE (desv.)': r['test_rmse'].std()}
t_cv0 = pd.DataFrame({k: cv_reg(m) for k, m in cand_r.items()}).T.round(3)
t_cv0.to_csv('resultados/reg_cv_sin_optimizar.csv')
display(t_cv0)

,MAE,RMSE,R2,RMSE (desv.)
Línea base (mediana),0.767,1.159,-0.001,0.007
Ridge,0.795,1.112,0.078,0.011
Random Forest,0.567,0.879,0.423,0.024
Gradient Boosting,0.629,0.896,0.402,0.024


### 2.2 Optimización de hiperparámetros (búsqueda en cuadrícula, 5 pliegues, criterio RMSE)

In [5]:
grids_r = {
    'Ridge': (pipe(Ridge()), {'m__alpha': [0.01, 0.1, 1, 10, 100, 1000]}),
    'Random Forest': (pipe(RandomForestRegressor(random_state=SEED, n_jobs=-1), escalar=False),
                      {'m__n_estimators': [200], 'm__max_depth': [6, 12, None], 'm__min_samples_leaf': [1, 5, 15]}),
    'Gradient Boosting': (pipe(GradientBoostingRegressor(random_state=SEED), escalar=False),
                          {'m__n_estimators': [100, 200], 'm__max_depth': [2, 3, 4], 'm__learning_rate': [0.05, 0.1]}),
}
best_r, filas = {}, []
for nombre, (p, g) in grids_r.items():
    gs = GridSearchCV(p, g, cv=cv_r, scoring='neg_root_mean_squared_error', n_jobs=-1).fit(Xtr, ytr_r)
    best_r[nombre + ' (optimizado)'] = gs.best_estimator_
    filas.append({'Modelo': nombre, 'Mejores hiperparámetros': {k.replace('m__',''): v for k, v in gs.best_params_.items()},
                  'RMSE CV antes': t_cv0.loc[nombre, 'RMSE'], 'RMSE CV después': round(-gs.best_score_, 3)})
t_opt_r = pd.DataFrame(filas).set_index('Modelo')
t_opt_r.to_csv('resultados/reg_optimizacion.csv')
pd.set_option('display.max_colwidth', 120)
display(t_opt_r)

,Mejores hiperparámetros,RMSE CV antes,RMSE CV después
Modelo,,,
Ridge,{'alpha': 100},1.112,1.112
Random Forest,"{'max_depth': 12, 'min_samples_leaf': 1, 'n_estimators': 200}",0.879,0.853
Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 200}",0.896,0.856


### 2.3 Evaluación final en prueba (una sola vez)

In [6]:
final_r = {'Línea base (mediana)': cand_r['Línea base (mediana)'].fit(Xtr, ytr_r)}
final_r.update(best_r)
filas, pred_r = [], {}
for k, m in final_r.items():
    p = m.predict(Xte); pred_r[k] = p
    filas.append({'Modelo': k, 'MAE': mean_absolute_error(yte_r, p),
                  'RMSE': mean_squared_error(yte_r, p) ** .5, 'R2': r2_score(yte_r, p)})
t_test_r = pd.DataFrame(filas).set_index('Modelo').round(3)
t_test_r.to_csv('resultados/reg_prueba.csv')
print(f'n prueba = {len(Xte)}'); display(t_test_r)

n prueba = 1368


,MAE,RMSE,R2
Modelo,,,
Línea base (mediana),0.769,1.185,-0.000
Ridge (optimizado),0.782,1.115,0.114
Random Forest (optimizado),0.543,0.845,0.491
Gradient Boosting (optimizado),0.574,0.851,0.484


In [7]:
# Figura 1: MAE y RMSE por modelo
fig, ax = plt.subplots(figsize=(8, 4.2))
w = 0.2; xs = np.arange(2); cols = [GRIS, VERDE, NARANJA, AZUL]
for i, (k, row) in enumerate(t_test_r.iterrows()):
    b = ax.bar(xs + (i - 1.5) * w, [row['MAE'], row['RMSE']], w, label=k, color=cols[i])
    ax.bar_label(b, fmt='%.2f', fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels(['MAE', 'RMSE']); ax.set_ylabel('Error (unidades de VEI)')
ax.set_title('Regresión: errores en datos de prueba (menor es mejor)'); ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig('figuras/reg_barras_mae_rmse.png'); plt.show()

# Figura 2: real vs predicción y residuos del mejor modelo (por RMSE)
mejor_r = t_test_r.drop(index='Línea base (mediana)')['RMSE'].idxmin(); p = pred_r[mejor_r]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(yte_r + np.random.default_rng(SEED).uniform(-.12, .12, len(yte_r)), p, s=8, alpha=.4, color=AZUL)
ax[0].plot([0, 7], [0, 7], 'k--', lw=1); ax[0].set_xlabel('VEI real (con ligera dispersión visual)'); ax[0].set_ylabel('VEI predicho')
ax[0].set_title('Real frente a predicción')
ax[1].scatter(p, yte_r - p, s=8, alpha=.4, color=NARANJA); ax[1].axhline(0, color='k', lw=1)
ax[1].set_xlabel('VEI predicho'); ax[1].set_ylabel('Residuo (real − predicho)'); ax[1].set_title('Residuos frente a predicción')
fig.suptitle(f'{mejor_r} · datos de prueba (n = {len(yte_r)})', fontsize=10)
plt.tight_layout(); plt.savefig('figuras/reg_real_vs_pred_residuos.png'); plt.show()
print('Mejor modelo de regresión:', mejor_r)

Mejor modelo de regresión: Random Forest (optimizado)


## 3. Clasificación: VEI < 3 frente a VEI ≥ 3
### 3.1 Modelos candidatos y validación cruzada estratificada (solo entrenamiento)

In [8]:
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cand_c = {
    'Línea base (clase mayoritaria)': pipe(DummyClassifier(strategy='most_frequent')),
    'Regresión logística':            pipe(LogisticRegression(max_iter=1000)),
    'Regresión logística balanceada': pipe(LogisticRegression(max_iter=1000, class_weight='balanced')),
    'Random Forest':      pipe(RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1), escalar=False),
    'Gradient Boosting':  pipe(GradientBoostingClassifier(random_state=SEED), escalar=False),
}
def cv_clf(m):
    r = cross_validate(m, Xtr, ytr_c, cv=cv_c, n_jobs=-1,
                       scoring=['accuracy', 'precision', 'recall', 'f1'])
    return {'Accuracy': r['test_accuracy'].mean(), 'Precision': r['test_precision'].mean(),
            'Recall': r['test_recall'].mean(), 'F1': r['test_f1'].mean(), 'F1 (desv.)': r['test_f1'].std()}
t_cv0_c = pd.DataFrame({k: cv_clf(m) for k, m in cand_c.items()}).T.round(3)
t_cv0_c.to_csv('resultados/clf_cv_sin_optimizar.csv'); display(t_cv0_c)

,Accuracy,Precision,Recall,F1,F1 (desv.)
Línea base (clase mayoritaria),0.762,0.000,0.000,0.000,0.000
Regresión logística,0.771,0.574,0.159,0.249,0.007
Regresión logística balanceada,0.732,0.452,0.578,0.507,0.013
Random Forest,0.838,0.680,0.603,0.638,0.024
Gradient Boosting,0.847,0.767,0.515,0.615,0.023


### 3.2 Optimización de hiperparámetros (cuadrícula, 5 pliegues, criterio F1)

In [9]:
grids_c = {
    'Regresión logística balanceada': (pipe(LogisticRegression(max_iter=2000, class_weight='balanced')),
                                       {'m__C': [0.01, 0.1, 1, 10, 100]}),
    'Random Forest': (pipe(RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight='balanced_subsample'), escalar=False),
                      {'m__n_estimators': [200], 'm__max_depth': [6, 12, None], 'm__min_samples_leaf': [1, 5, 15]}),
    'Gradient Boosting': (pipe(GradientBoostingClassifier(random_state=SEED), escalar=False),
                          {'m__n_estimators': [100, 200], 'm__max_depth': [2, 3, 4], 'm__learning_rate': [0.05, 0.1]}),
}
best_c, filas = {}, []
for nombre, (p, g) in grids_c.items():
    gs = GridSearchCV(p, g, cv=cv_c, scoring='f1', n_jobs=-1).fit(Xtr, ytr_c)
    best_c[nombre + ' (optimizado)'] = gs.best_estimator_
    antes = t_cv0_c.loc[nombre, 'F1']
    filas.append({'Modelo': nombre, 'Mejores hiperparámetros': {k.replace('m__',''): v for k, v in gs.best_params_.items()},
                  'F1 CV antes': antes, 'F1 CV después': round(gs.best_score_, 3)})
t_opt_c = pd.DataFrame(filas).set_index('Modelo'); t_opt_c.to_csv('resultados/clf_optimizacion.csv'); display(t_opt_c)

,Mejores hiperparámetros,F1 CV antes,F1 CV después
Modelo,,,
Regresión logística balanceada,{'C': 1},0.507,0.507
Random Forest,"{'max_depth': 12, 'min_samples_leaf': 1, 'n_estimators': 200}",0.638,0.668
Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 200}",0.615,0.645


### 3.3 Ajuste del umbral de decisión
El umbral 0.5 no es sagrado. Se elige el umbral que maximiza F1 usando **probabilidades fuera de pliegue calculadas solo con entrenamiento**; la prueba no interviene.

In [10]:
mejor_c_nombre = max(best_c, key=lambda k: t_opt_c.loc[k.replace(' (optimizado)', ''), 'F1 CV después'])
print('Modelo base para ajustar el umbral:', mejor_c_nombre)
from sklearn.base import clone
proba_oof = cross_val_predict(clone(best_c[mejor_c_nombre]), Xtr, ytr_c, cv=cv_c, method='predict_proba', n_jobs=-1)[:, 1]
ums = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(ytr_c, proba_oof >= u) for u in ums]
rec = [recall_score(ytr_c, proba_oof >= u) for u in ums]
pre = [precision_score(ytr_c, proba_oof >= u, zero_division=0) for u in ums]
UMBRAL = float(ums[int(np.argmax(f1s))])
print(f'Umbral elegido (máx. F1 en validación): {UMBRAL:.2f}  ->  F1={max(f1s):.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ums, pre, label='Precision', color=VERDE); ax.plot(ums, rec, label='Recall', color=NARANJA); ax.plot(ums, f1s, label='F1', color=AZUL)
ax.axvline(UMBRAL, color='k', ls='--', lw=1, label=f'Umbral elegido = {UMBRAL:.2f}'); ax.axvline(0.5, color=GRIS, ls=':', lw=1, label='Umbral 0.5')
ax.set_xlabel('Umbral de clasificación'); ax.set_ylabel('Métrica'); ax.set_title('Efecto del umbral (validación cruzada, solo entrenamiento)')
ax.legend(fontsize=8, frameon=False); plt.tight_layout(); plt.savefig('figuras/clf_curva_umbral.png'); plt.show()

Modelo base para ajustar el umbral: Random Forest (optimizado)


Umbral elegido (máx. F1 en validación): 0.53  ->  F1=0.674


### 3.4 Evaluación final en prueba (una sola vez)

In [11]:
final_c = {'Línea base (clase mayoritaria)': cand_c['Línea base (clase mayoritaria)'].fit(Xtr, ytr_c),
           'Regresión logística (umbral 0.5)': cand_c['Regresión logística'].fit(Xtr, ytr_c)}
final_c.update({k + ' (umbral 0.5)': v for k, v in best_c.items()})
filas, pred_c = [], {}
def fila(nombre, yhat):
    pred_c[nombre] = yhat
    tn, fp, fn, tp = confusion_matrix(yte_c, yhat).ravel()
    return {'Modelo': nombre, 'Accuracy': accuracy_score(yte_c, yhat), 'Precision': precision_score(yte_c, yhat, zero_division=0),
            'Recall': recall_score(yte_c, yhat), 'F1': f1_score(yte_c, yhat), 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}
for k, m in final_c.items(): filas.append(fila(k, m.predict(Xte)))
# Modelo seleccionado con umbral ajustado
sel = best_c[mejor_c_nombre]
nombre_sel = f'{mejor_c_nombre.replace(" (optimizado)", "")} (umbral {UMBRAL:.2f})'
filas.append(fila(nombre_sel, (sel.predict_proba(Xte)[:, 1] >= UMBRAL).astype(int)))
t_test_c = pd.DataFrame(filas).set_index('Modelo')
t_test_c[['Accuracy', 'Precision', 'Recall', 'F1']] = t_test_c[['Accuracy', 'Precision', 'Recall', 'F1']].round(3)
t_test_c.to_csv('resultados/clf_prueba.csv')
print(f'n prueba = {len(Xte)} | positivos = {yte_c.sum()}'); display(t_test_c)

n prueba = 1368 | positivos = 325


,Accuracy,Precision,Recall,F1,TN,FP,FN,TP
Modelo,,,,,,,,
Línea base (clase mayoritaria),0.762,0.000,0.000,0.000,1043,0,325,0
Regresión logística (umbral 0.5),0.781,0.647,0.169,0.268,1013,30,270,55
Regresión logística balanceada (optimizado) (umbral 0.5),0.740,0.463,0.594,0.520,819,224,132,193
Random Forest (optimizado) (umbral 0.5),0.865,0.732,0.680,0.705,962,81,104,221
Gradient Boosting (optimizado) (umbral 0.5),0.871,0.811,0.594,0.686,998,45,132,193
Random Forest (umbral 0.53),0.866,0.752,0.652,0.699,973,70,113,212


In [12]:
# Figura: barras de métricas
met = ['Accuracy', 'Precision', 'Recall', 'F1']
fig, ax = plt.subplots(figsize=(10, 4.4))
sub = t_test_c.loc[['Línea base (clase mayoritaria)', 'Regresión logística (umbral 0.5)', nombre_sel]]
cols = [GRIS, VERDE, NARANJA]; w = .26
for i, (k, row) in enumerate(sub.iterrows()):
    b = ax.bar(np.arange(4) + (i - 1) * w, row[met].astype(float), w, label=k, color=cols[i]); ax.bar_label(b, fmt='%.2f', fontsize=8)
ax.set_xticks(range(4)); ax.set_xticklabels(met); ax.set_ylim(0, 1.05); ax.set_ylabel('Valor de la métrica')
ax.set_title('Clasificación: línea base, modelo inicial y modelo optimizado (datos de prueba)'); ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig('figuras/clf_barras_metricas.png'); plt.show()

# Matrices de confusión: inicial vs optimizado
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, k in zip(ax, ['Regresión logística (umbral 0.5)', nombre_sel]):
    ConfusionMatrixDisplay(confusion_matrix(yte_c, pred_c[k]), display_labels=['VEI < 3', 'VEI ≥ 3']).plot(ax=a, colorbar=False, cmap='Greens')
    a.set_title(k, fontsize=9)
fig.suptitle(f'Matrices de confusión en prueba (n = {len(yte_c)})', fontsize=10)
plt.tight_layout(); plt.savefig('figuras/clf_matrices_confusion.png'); plt.show()

## 4. Importancia de variables y resumen

In [13]:
rf = [v for k, v in best_c.items() if 'Random Forest' in k][0].named_steps['m']
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(6, 3)); ax.barh(imp.index, imp.values, color=VERDE)
ax.set_xlabel('Importancia (Random Forest optimizado, ajustado con entrenamiento)'); ax.set_title('Importancia de variables de entrada')
plt.tight_layout(); plt.savefig('figuras/importancia_variables.png'); plt.show()
print(imp.round(3).to_string())

resumen = {'n_total': int(len(df)), 'n_modelado': int(len(d)), 'n_train': int(len(Xtr)), 'n_test': int(len(Xte)),
           'positivos_test': int(yte_c.sum()), 'umbral_elegido': UMBRAL, 'mejor_regresion': mejor_r, 'modelo_clasificacion': nombre_sel}
json.dump(resumen, open('resultados/resumen.json', 'w'), indent=2, ensure_ascii=False); print(resumen)

elevacion_m    0.137
longitud       0.175
latitud        0.209
anio_inicio    0.479
{'n_total': 9918, 'n_modelado': 6840, 'n_train': 5472, 'n_test': 1368, 'positivos_test': 325, 'umbral_elegido': 0.5299999999999999, 'mejor_regresion': 'Random Forest (optimizado)', 'modelo_clasificacion': 'Random Forest (umbral 0.53)'}


## 5. Prueba de robustez: dividir por volcán
Un mismo volcán aparece en varias erupciones y sus coordenadas y elevación son idénticas. Con la división aleatoria, un árbol puede "reconocer" un volcán visto en entrenamiento. Para medir la capacidad de generalizar a **volcanes nuevos**, se repite la evaluación con una división donde todas las erupciones de un volcán quedan del mismo lado (`GroupShuffleSplit` por `volcano_number`), con los mismos hiperparámetros ya optimizados.

In [14]:
from sklearn.model_selection import GroupShuffleSplit
grupos = d['volcano_number']
print('Volcanes distintos:', grupos.nunique(), '| mediana de erupciones por volcán:', int(d.groupby('volcano_number').size().median()))
g_tr, g_te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y_reg, groups=grupos))
Xg_tr, Xg_te = X.iloc[g_tr], X.iloc[g_te]
rob = []
# Regresión
mr = clone(best_r['Random Forest (optimizado)']).fit(Xg_tr, y_reg.iloc[g_tr]); br = clone(cand_r['Línea base (mediana)']).fit(Xg_tr, y_reg.iloc[g_tr])
for nombre, m in [('Línea base (mediana)', br), ('Random Forest (optimizado)', mr)]:
    p = m.predict(Xg_te)
    rob.append({'Tarea': 'Regresión', 'División': 'Por volcán', 'Modelo': nombre, 'MAE': mean_absolute_error(y_reg.iloc[g_te], p),
                'RMSE': mean_squared_error(y_reg.iloc[g_te], p) ** .5, 'R2': r2_score(y_reg.iloc[g_te], p)})
pr = pred_r['Random Forest (optimizado)']
rob.append({'Tarea': 'Regresión', 'División': 'Aleatoria', 'Modelo': 'Random Forest (optimizado)', 'MAE': mean_absolute_error(yte_r, pr),
            'RMSE': mean_squared_error(yte_r, pr) ** .5, 'R2': r2_score(yte_r, pr)})
# Clasificación (umbral 0.5)
mc = clone(best_c['Random Forest (optimizado)']).fit(Xg_tr, y_clf.iloc[g_tr]); yg = y_clf.iloc[g_te]
pc = mc.predict(Xg_te)
rob.append({'Tarea': 'Clasificación', 'División': 'Por volcán', 'Modelo': 'Random Forest (optimizado)', 'Accuracy': accuracy_score(yg, pc),
            'Precision': precision_score(yg, pc), 'Recall': recall_score(yg, pc), 'F1': f1_score(yg, pc)})
pa = pred_c['Random Forest (optimizado) (umbral 0.5)']
rob.append({'Tarea': 'Clasificación', 'División': 'Aleatoria', 'Modelo': 'Random Forest (optimizado)', 'Accuracy': accuracy_score(yte_c, pa),
            'Precision': precision_score(yte_c, pa), 'Recall': recall_score(yte_c, pa), 'F1': f1_score(yte_c, pa)})
t_rob = pd.DataFrame(rob).round(3); t_rob.to_csv('resultados/robustez_por_volcan.csv', index=False)
print(f'Prueba por volcán: {len(g_te)} erupciones, {int(yg.sum())} positivos'); display(t_rob)

Volcanes distintos: 645 | mediana de erupciones por volcán: 4


Prueba por volcán: 1428 erupciones, 335 positivos


,Tarea,División,Modelo,MAE,RMSE,R2,Accuracy,Precision,Recall,F1
0,Regresión,Por volcán,Línea base (mediana),0.842,1.234,-0.008,NaN,NaN,NaN,NaN
1,Regresión,Por volcán,Random Forest (optimizado),0.716,1.008,0.327,NaN,NaN,NaN,NaN
2,Regresión,Aleatoria,Random Forest (optimizado),0.543,0.845,0.491,NaN,NaN,NaN,NaN
3,Clasificación,Por volcán,Random Forest (optimizado),NaN,NaN,NaN,0.840,0.689,0.576,0.628
4,Clasificación,Aleatoria,Random Forest (optimizado),NaN,NaN,NaN,0.865,0.732,0.680,0.705


In [15]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
r = t_rob[t_rob.Tarea == 'Regresión'].set_index(['División', 'Modelo'])['R2']
et = ['Aleatoria\nRF', 'Por volcán\nRF', 'Por volcán\nlínea base']
vals = [r[('Aleatoria', 'Random Forest (optimizado)')], r[('Por volcán', 'Random Forest (optimizado)')], r[('Por volcán', 'Línea base (mediana)')]]
b = ax[0].bar(et, vals, color=[NARANJA, AZUL, GRIS]); ax[0].bar_label(b, fmt='%.2f'); ax[0].set_ylabel('R²'); ax[0].set_title('Regresión: R² en prueba')
c = t_rob[t_rob.Tarea == 'Clasificación'].set_index('División')['F1']
b = ax[1].bar(['Aleatoria\nRF', 'Por volcán\nRF'], [c['Aleatoria'], c['Por volcán']], color=[NARANJA, AZUL]); ax[1].bar_label(b, fmt='%.2f')
ax[1].set_ylabel('F1'); ax[1].set_ylim(0, 1); ax[1].set_title('Clasificación: F1 en prueba (umbral 0.5)')
plt.tight_layout(); plt.savefig('figuras/robustez_division_por_volcan.png'); plt.show()